功能：读取当前目录 `ClusterGAGG.root` 中 `SimulatedTree` 的 `ClusterGAGG.fClusterGAGGHits`，按 ROOT entry 顺序打印前 5 个有击中的事件及每个事件的全部 hit 字段。

方法：用 uproot 读取该结构体集合的所有拆分字段，awkward 保留“事件 → hit”的变长结构；每批读取 1000 个事件，跳过空集合，打印满 5 个非空事件后停止。

注意事项：需要安装 uproot、awkward，并在本 Notebook 所在目录运行。entry 从 0 开始，与 hit 内的 EventID 分别显示；不筛选粒子、晶体或 ClusterGAGGType。Energy 单位为 MeV，LengthInGAGG 单位为 mm。所有字段逐行原样打印，不展平事件、不训练模型、不写入 ROOT 文件；不足 5 个非空事件时打印实际数量。


In [1]:
from pathlib import Path

import awkward as ak
import uproot

ROOT_PATH = Path("ClusterGAGG.root")
TREE_NAME = "SimulatedTree"
HIT_BRANCH = "ClusterGAGG/fClusterGAGGHits"  # uproot 用 / 访问 ROOT 分支层级。
MAX_EVENTS = 5
CHUNK_SIZE = 1000


In [2]:
printed_events = 0
units = {"Energy": "MeV", "LengthInGAGG": "mm"}

with uproot.open(ROOT_PATH) as root_file:
    tree = root_file[TREE_NAME]
    hit_branch = tree[HIT_BRANCH]
    # 从实际分支获取全部 hit 字段，保持 ROOT 中的字段顺序。
    field_branches = {
        branch.name.rsplit(".", 1)[-1]: branch
        for branch in hit_branch.branches
    }

    for entry_start in range(0, tree.num_entries, CHUNK_SIZE):
        entry_stop = min(entry_start + CHUNK_SIZE, tree.num_entries)
        hit_arrays = {
            field: branch.array(
                entry_start=entry_start, entry_stop=entry_stop, library="ak"
            )
            for field, branch in field_branches.items()
        }
        hit_counts = ak.num(hit_arrays["EventID"], axis=1)
        nonempty_entries = ak.to_list(ak.where(hit_counts > 0)[0])

        for local_entry in nonempty_entries:
            entry = entry_start + local_entry
            hit_count = int(hit_counts[local_entry])
            print(f"Event {printed_events + 1}: entry={entry}, hits={hit_count}")

            for hit_index in range(hit_count):
                print(f"  Hit {hit_index}:")
                for field, values in hit_arrays.items():
                    value = ak.to_list(values[local_entry][hit_index])
                    unit = f" {units[field]}" if field in units else ""
                    print(f"    {field} = {value}{unit}")
            print()

            printed_events += 1
            if printed_events == MAX_EVENTS:
                break

        if printed_events == MAX_EVENTS:
            break

if printed_events < MAX_EVENTS:
    print(f"文件中仅找到 {printed_events} 个有击中的事件。")


Event 1: entry=2, hits=1
  Hit 0:
    EventID = 2
    CrystalNbr = 31
    ParticleName = triton
    TrackID = 1
    Energy = 25.822669172979037 MeV
    LengthInGAGG = 13.927528602414585 mm
    ClusterGAGGType = ClusterGAGG
    PunchThroughCrystal = True

Event 2: entry=17, hits=2
  Hit 0:
    EventID = 17
    CrystalNbr = 34
    ParticleName = triton
    TrackID = 1
    Energy = 380.03183627045235 MeV
    LengthInGAGG = 121.67964051645794 mm
    ClusterGAGGType = ClusterGAGG
    PunchThroughCrystal = True
  Hit 1:
    EventID = 17
    CrystalNbr = 1
    ParticleName = triton
    TrackID = 1
    Energy = 73.9951239938752 MeV
    LengthInGAGG = 35.55992074412839 mm
    ClusterGAGGType = DeltaEGAGGRight
    PunchThroughCrystal = True

Event 3: entry=23, hits=2
  Hit 0:
    EventID = 23
    CrystalNbr = 15
    ParticleName = triton
    TrackID = 1
    Energy = 170.6582256258067 MeV
    LengthInGAGG = 67.728394581784 mm
    ClusterGAGGType = ClusterGAGG
    PunchThroughCrystal = True
  Hit 